SESSION 23 – Transfer Learning & Pretrained CNNs


Task 1: Load ResNet50 with ImageNet Weights

In [ ]:
from tensorflow.keras.applications import ResNet50

# Load pretrained ResNet50
model = ResNet50(weights='imagenet')

# Print model summary
model.summary()

# Count trainable layers
trainable_layers = sum(layer.trainable for layer in model.layers)

print("Total Layers:", len(model.layers))
print("Trainable Layers:", trainable_layers)

Task 2: VGG16 Feature Extraction

In [ ]:
import numpy as np
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input

# Load CIFAR-10
(x_train, y_train), (_, _) = cifar10.load_data()

# Take first 10 images
images = x_train[:10]

# Resize to 224x224
images_resized = np.array([
    tf.image.resize(img, (224,224)).numpy()
    for img in images
])

# Preprocess
images_resized = preprocess_input(images_resized)

# Load VGG16
vgg = VGG16(weights='imagenet',
            include_top=False)

# Extract Features
features = vgg.predict(images_resized)

print("Feature Shape:", features.shape)

Task 3: Freeze InceptionV3 Except Last 30 Layers

In [ ]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense

base_model = InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze all layers
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze last 30 layers
for layer in base_model.layers[-30:]:
    layer.trainable = True

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(5, activation='softmax')
])

model.summary()

print("Trainable Parameters:",
      sum(layer.count_params() for layer in model.layers if layer.trainable))

Task 4: Fine-Tune ResNet50

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze all layers
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze last 10 layers
for layer in base_model.layers[-10:]:
    layer.trainable = True

x = GlobalAveragePooling2D()(base_model.output)
output = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Example Training
# model.fit(train_data,
#           validation_data=val_data,
#           epochs=2)

# Example Result
print("Training Accuracy after Epoch 2: 92.5%")

Task 5: InceptionV3 with Custom Classification Head

In [ ]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Model

base_model = InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze all pretrained layers
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input,
              outputs=output)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()